# Convert `comparisons_berlin(2).p` to `train.py`-compatible pickle + overlap report

This notebook does four things:

1. Loads the legacy pickle `comparisons_berlin(2).p` (created with an older pandas) using compatibility shims.
2. Normalizes it into the schema expected by your training pipeline (`train.py` / `data.py`):
   - `score`, `dataset`, `image_l`, `image_r`, `has_eyetracker`, `npy_file_l`, `npy_file_r`, `survey_id`, `trial_id`
3. Writes a new pickle: `comparisons_berlin_for_train.pickle`
4. Compares overlap with your existing `comparisons_df.pickle` (exact-order and order-invariant).

**Assumptions**
- The legacy dataframe contains columns similar to:
  `user`, `score`, `dataset`, and either (`image_i`,`image_j`) or (`image_l`,`image_r`).
- Scores are in {-1, 0, +1}.


In [1]:
# =========================================
# 1) Imports & Paths
# =========================================
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
import sys

# INPUTS
IN_PICKLE = Path("comparisons_berlin(2).p")     # legacy file you downloaded
REF_PICKLE = Path("comparisons_df.pickle")      # existing training comparisons (reference)

# OUTPUT
OUT_PICKLE = Path("comparisons_berlin_for_train.pickle")

assert IN_PICKLE.exists(), f"Missing input file: {IN_PICKLE.resolve()}"
print("IN_PICKLE :", IN_PICKLE.resolve())
print("REF_PICKLE:", REF_PICKLE.resolve(), "(exists)" if REF_PICKLE.exists() else "(NOT FOUND)")
print("OUT_PICKLE:", OUT_PICKLE.resolve())
print("pandas version:", pd.__version__)


IN_PICKLE : /home/csantiago/comparisons_berlin(2).p
REF_PICKLE: /home/csantiago/comparisons_df.pickle (exists)
OUT_PICKLE: /home/csantiago/comparisons_berlin_for_train.pickle
pandas version: 2.3.3


In [2]:
# =========================================
# 2) Load legacy pickle with pandas-compat shims
# =========================================
# Many older pandas pickles reference internal modules/classes that moved/vanished in new pandas.
# The following shims make the unpickling succeed without changing your environment.

obj = pd.read_pickle(IN_PICKLE)

print("Loaded type:", type(obj))
print("Shape:", obj.shape)
display(obj.head(10))


Loaded type: <class 'pandas.core.frame.DataFrame'>
Shape: (7281, 12)


,index,datetime,user,image_i,image_j,score,dataset,image_l,image_r,Winner,Loser,Tie
0,406,2022-09-06 17:13:23,cycling9334a308469b956854470ed3668c578f7c99fa3...,berlin/209.jpg,berlin/7819.jpg,1,berlin,209,7819,7819,209,0
1,407,2022-09-06 17:13:33,cycling9334a308469b956854470ed3668c578f7c99fa3...,berlin/210.jpg,berlin/2123.jpg,1,berlin,210,2123,2123,210,0
2,408,2022-09-06 17:13:43,cycling9334a308469b956854470ed3668c578f7c99fa3...,berlin/211.jpg,berlin/5265.jpg,-1,berlin,211,5265,211,5265,0
3,409,2022-09-06 17:13:52,cycling9334a308469b956854470ed3668c578f7c99fa3...,berlin/212.jpg,berlin/2024.jpg,-1,berlin,212,2024,212,2024,0
4,410,2022-09-06 17:14:10,cycling9334a308469b956854470ed3668c578f7c99fa3...,berlin/213.jpg,berlin/9692.jpg,1,berlin,213,9692,9692,213,0
5,411,2022-09-06 17:14:16,cycling9334a308469b956854470ed3668c578f7c99fa3...,berlin/214.jpg,berlin/5060.jpg,-1,berlin,214,5060,214,5060,0
6,412,2022-09-06 17:14:19,cycling9334a308469b956854470ed3668c578f7c99fa3...,berlin/216.jpg,berlin/1289.jpg,-1,berlin,216,1289,216,1289,0
7,413,2022-09-06 17:14:29,cycling9334a308469b956854470ed3668c578f7c99fa3...,berlin/217.jpg,berlin/5195.jpg,0,berlin,217,5195,5195,217,1
8,414,2022-09-06 17:14:44,cycling9334a308469b956854470ed3668c578f7c99fa3...,berlin/116.jpg,berlin/218.jpg,0,berlin,116,218,218,116,1
9,415,2022-09-06 17:15:04,cycling9334a308469b956854470ed3668c578f7c99fa3...,berlin/219.jpg,berlin/8116.jpg,0,berlin,219,8116,8116,219,1


In [3]:
# =========================================
# 3) Inspect schema (sanity)
# =========================================
print("Columns:", list(obj.columns))
display(obj.dtypes)

# Minimal checks for expected legacy columns
for c in ["user", "score", "dataset", "image_i", "image_j", "image_l", "image_r"]:
    print(f"{c:>10} present:", c in obj.columns)


Columns: ['index', 'datetime', 'user', 'image_i', 'image_j', 'score', 'dataset', 'image_l', 'image_r', 'Winner', 'Loser', 'Tie']


index        int64
datetime    object
user        object
image_i     object
image_j     object
score        int64
dataset     object
image_l     object
image_r     object
Winner      object
Loser       object
Tie          int64
dtype: object

      user present: True
     score present: True
   dataset present: True
   image_i present: True
   image_j present: True
   image_l present: True
   image_r present: True


In [4]:
# =========================================
# 4) Helpers: normalize dataset + image identifiers
# =========================================
from pathlib import Path

def to_dataset_name(x):
    """Normalize dataset value to a clean string."""
    if pd.isna(x):
        return "__unknown__"
    return str(x).strip()

def to_filename_jpg(x):
    """
    Convert an image identifier into a filename with .jpg.
    Handles:
      - numeric ids: 209 -> "209.jpg"
      - strings: "berlin/209.jpg" -> "209.jpg"
      - strings without extension: "209" -> "209.jpg"
    """
    if pd.isna(x):
        return None

    # numeric-like -> int -> "<id>.jpg"
    try:
        xi = int(float(x))
        return f"{xi}.jpg"
    except Exception:
        pass

    s = str(x).strip()
    base = Path(s).name  # strips any 'berlin/' prefix
    if "." not in base:
        base = base + ".jpg"
    return base


In [5]:
# =========================================
# 5) Build train.py/data.py compatible dataframe
# =========================================
# Required columns for your pipeline:
#   score (int in {-1,0,1})
#   dataset (string folder name under images/)
#   image_l, image_r (filenames like '209.jpg')
# Optional but expected by your pipeline code paths:
#   has_eyetracker (bool)
#   npy_file_l, npy_file_r (None for non-eyetracker rows)
#   survey_id (user/session identifier)
#   trial_id (index within a survey_id)

df_train = pd.DataFrame()

# 5.1 Score
df_train["score"] = pd.to_numeric(obj["score"], errors="coerce")
df_train = df_train[df_train["score"].isin([-1, 0, 1])].copy()
df_train["score"] = df_train["score"].astype(int)

# 5.2 Dataset
if "dataset" in obj.columns:
    df_train["dataset"] = obj["dataset"].apply(to_dataset_name)
else:
    df_train["dataset"] = "berlin"  # conservative fallback

# 5.3 Images
# Prefer image_i/image_j if present (often looks like 'berlin/209.jpg').
if "image_i" in obj.columns and "image_j" in obj.columns:
    df_train["image_l"] = obj["image_i"].apply(to_filename_jpg)
    df_train["image_r"] = obj["image_j"].apply(to_filename_jpg)
elif "image_l" in obj.columns and "image_r" in obj.columns:
    df_train["image_l"] = obj["image_l"].apply(to_filename_jpg)
    df_train["image_r"] = obj["image_r"].apply(to_filename_jpg)
else:
    raise ValueError("Could not find either (image_i,image_j) or (image_l,image_r) in legacy dataframe.")

# 5.4 Eyetracker-related fields (this file is survey comparisons; no gaze maps)
df_train["has_eyetracker"] = False
df_train["npy_file_l"] = None
df_train["npy_file_r"] = None

# 5.5 Survey and trial identifiers
if "user" in obj.columns:
    df_train["survey_id"] = obj["user"].astype(str).str.strip()
else:
    df_train["survey_id"] = "unknown"

df_train["trial_id"] = df_train.groupby("survey_id").cumcount().astype(int)

# 5.6 Drop any rows with missing essentials
before = len(df_train)
df_train = df_train.dropna(subset=["dataset", "image_l", "image_r", "survey_id"]).copy()
after = len(df_train)

print(f"Dropped {before-after} rows with missing essentials. Remaining: {after}")
print("df_train shape:", df_train.shape)
display(df_train.head(10))

# 5.7 Write pickle
df_train.to_pickle(OUT_PICKLE)
print("Saved:", OUT_PICKLE.resolve())


Dropped 0 rows with missing essentials. Remaining: 7281
df_train shape: (7281, 9)


,score,dataset,image_l,image_r,has_eyetracker,npy_file_l,npy_file_r,survey_id,trial_id
0,1,berlin,209.jpg,7819.jpg,False,None,None,cycling9334a308469b956854470ed3668c578f7c99fa3...,0
1,1,berlin,210.jpg,2123.jpg,False,None,None,cycling9334a308469b956854470ed3668c578f7c99fa3...,1
2,-1,berlin,211.jpg,5265.jpg,False,None,None,cycling9334a308469b956854470ed3668c578f7c99fa3...,2
3,-1,berlin,212.jpg,2024.jpg,False,None,None,cycling9334a308469b956854470ed3668c578f7c99fa3...,3
4,1,berlin,213.jpg,9692.jpg,False,None,None,cycling9334a308469b956854470ed3668c578f7c99fa3...,4
5,-1,berlin,214.jpg,5060.jpg,False,None,None,cycling9334a308469b956854470ed3668c578f7c99fa3...,5
6,-1,berlin,216.jpg,1289.jpg,False,None,None,cycling9334a308469b956854470ed3668c578f7c99fa3...,6
7,0,berlin,217.jpg,5195.jpg,False,None,None,cycling9334a308469b956854470ed3668c578f7c99fa3...,7
8,0,berlin,116.jpg,218.jpg,False,None,None,cycling9334a308469b956854470ed3668c578f7c99fa3...,8
9,0,berlin,219.jpg,8116.jpg,False,None,None,cycling9334a308469b956854470ed3668c578f7c99fa3...,9


Saved: /home/csantiago/comparisons_berlin_for_train.pickle


In [6]:
# =========================================
# 6) Statistics (similar to your non-eyetracker stats)
# =========================================
print("\n==================== NEW PICKLE OVERVIEW ====================")
print("Total comparisons:", len(df_train))
print("Unique users      :", df_train["survey_id"].nunique())
print("Datasets          :", df_train["dataset"].nunique())
print("Dataset counts    :")
display(df_train["dataset"].value_counts().head(30))

print("\nScore distribution:")
display(df_train["score"].value_counts().reindex([-1,0,1], fill_value=0).to_frame("count"))

print("\nAverage image appearances (within each dataset):")
rows = []
for dset, g in df_train.groupby("dataset"):
    counts = g["image_l"].value_counts().add(g["image_r"].value_counts(), fill_value=0)
    rows.append({
        "dataset": dset,
        "n_comparisons": len(g),
        "n_unique_images": int(counts.shape[0]),
        "avg_appearances_per_image": float(counts.mean()),
        "median_appearances_per_image": float(counts.median()),
        "max_appearances_single_image": int(counts.max()),
        "min_appearances_single_image": int(counts.min()),
    })
stats_images = pd.DataFrame(rows).sort_values("n_comparisons", ascending=False)
display(stats_images.head(30))



==================== NEW PICKLE OVERVIEW ====================
Total comparisons: 7281
Unique users      : 251
Datasets          : 1
Dataset counts    :


dataset
berlin    7281
Name: count, dtype: int64


Score distribution:


,count
score,
-1,2907
0,1369
1,3005



Average image appearances (within each dataset):


,dataset,n_comparisons,n_unique_images,avg_appearances_per_image,median_appearances_per_image,max_appearances_single_image,min_appearances_single_image
0,berlin,7281,4481,3.249721,3.0,24,1


In [7]:
# =========================================
# 7) Overlap vs comparisons_df.pickle
# =========================================
# This cell compares overlap in two ways:
#   A) Exact-order overlap: (dataset, image_l, image_r, score) must match exactly
#   B) Order-invariant overlap: swap-safe (dataset, min(image_l,image_r), max(...), score)

if not REF_PICKLE.exists():
    print("[INFO] Reference pickle not found. Skipping overlap.")
else:
    ref = pd.read_pickle(REF_PICKLE)

    needed = ["dataset", "image_l", "image_r", "score"]
    missing = [c for c in needed if c not in ref.columns]
    if missing:
        raise ValueError(f"Reference pickle missing required columns: {missing}")

    # Normalize reference to the same conventions
    ref_norm = ref.copy()
    ref_norm["dataset"] = ref_norm["dataset"].apply(to_dataset_name)
    ref_norm["image_l"] = ref_norm["image_l"].apply(to_filename_jpg)
    ref_norm["image_r"] = ref_norm["image_r"].apply(to_filename_jpg)
    ref_norm["score"] = pd.to_numeric(ref_norm["score"], errors="coerce")
    ref_norm = ref_norm[ref_norm["score"].isin([-1,0,1])].copy()
    ref_norm["score"] = ref_norm["score"].astype(int)

    key_cols = ["dataset", "image_l", "image_r", "score"]

    a = df_train[key_cols].dropna().drop_duplicates()
    b = ref_norm[key_cols].dropna().drop_duplicates()

    a_set = set(map(tuple, a.values.tolist()))
    b_set = set(map(tuple, b.values.tolist()))

    exact_intersection = a_set & b_set

    print("\n==================== OVERLAP REPORT ====================")
    print("A (new) unique comparisons (exact-order):", len(a_set))
    print("B (ref) unique comparisons (exact-order):", len(b_set))
    print("Exact-order overlap:", len(exact_intersection))
    print("Exact-order overlap % of A:", 100.0 * len(exact_intersection) / max(1, len(a_set)))

    # Order-invariant overlap
    def unordered_key(df):
        l = df["image_l"].astype(str)
        r = df["image_r"].astype(str)
        lo = np.minimum(l, r)
        hi = np.maximum(l, r)
        return list(zip(df["dataset"].astype(str), lo, hi, df["score"].astype(int)))

    a_u = set(unordered_key(a))
    b_u = set(unordered_key(b))
    u_intersection = a_u & b_u

    print("\nOrder-invariant (swap-safe) overlap:", len(u_intersection))
    print("Order-invariant overlap % of A:", 100.0 * len(u_intersection) / max(1, len(a_u)))

    print("\nSample overlaps (order-invariant) [up to 10]:")
    for t in list(u_intersection)[:10]:
        print(t)



==================== OVERLAP REPORT ====================
A (new) unique comparisons (exact-order): 7251
B (ref) unique comparisons (exact-order): 12918
Exact-order overlap: 7124
Exact-order overlap % of A: 98.24851744586954

Order-invariant (swap-safe) overlap: 7124
Order-invariant overlap % of A: 98.24851744586954

Sample overlaps (order-invariant) [up to 10]:
('berlin', '5948.jpg', '6319.jpg', -1)
('berlin', '1779.jpg', '7867.jpg', 1)
('berlin', '7283.jpg', '8184.jpg', 0)
('berlin', '6581.jpg', '7976.jpg', 0)
('berlin', '5296.jpg', '6604.jpg', 0)
('berlin', '5482.jpg', '8238.jpg', -1)
('berlin', '1534.jpg', '2816.jpg', -1)
('berlin', '4916.jpg', '867.jpg', 1)
('berlin', '335.jpg', '6869.jpg', 0)
('berlin', '4720.jpg', '7974.jpg', 1)
